# 🌾 Wheat Plant Disease: Baseline CNN Model Development
**Assigned Task:** Model Development (Sprint 2 - Part A)
**Author:** Sachin Choudhary (Team Lead, AI/ML)
**Dataset:** [`kushagra3204/wheat-plant-diseases`](https://www.kaggle.com/datasets/kushagra3204/wheat-plant-diseases)

---
### Workflow:
1. Install all required dependencies first.
2. Set up data augmentation & input generators.
3. Design Custom CNN architecture with `GlobalAveragePooling2D` (fixing the naive 246k-neuron Flatten flaw).
4. Train baseline model with Adam optimizer.
5. Plot training loss and accuracy progression curves.
6. Evaluate baseline validation accuracy.
7. Save `baseline_model.h5` and `baseline_history.json`.

In [ ]:
# Step 1: Install all required dependencies
!pip install -q tensorflow numpy pandas matplotlib seaborn pillow
print('Dependencies installed successfully!')


In [ ]:
# Step 2: Import libraries & check GPU
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Step 3: Find dataset path automatically
def find_data_path():
    candidates = [
        '/kaggle/input/wheat-plant-diseases/data',
        '/kaggle/input/wheat-plant-diseases',
        '/kaggle/input/data',
        './data'
    ]
    for p in candidates:
        if os.path.exists(os.path.join(p, 'train')):
            return os.path.join(p, 'train'), os.path.join(p, 'valid')
    for root, dirs, files in os.walk('/kaggle/input' if os.path.exists('/kaggle/input') else '.'):
        if 'train' in dirs and 'valid' in dirs:
            return os.path.join(root, 'train'), os.path.join(root, 'valid')
    return None, None

TRAIN_DIR, VALID_DIR = find_data_path()
print('Train Path:', TRAIN_DIR)
print('Valid Path:', VALID_DIR)


In [ ]:
# Step 4: Data Augmentation & Generators
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training generator with data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation generator (rescaling only)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print(f'Total Classes: {num_classes}')


In [ ]:
# Step 5: Build Baseline Custom CNN Architecture
baseline_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    
    # Global Average Pooling (replaces 246k-neuron Flatten to prevent overfitting)
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

baseline_model.summary()


In [ ]:
# Step 6: Train Baseline Custom CNN (10 Epochs)
print('Starting Baseline Model Training...')
history_baseline = baseline_model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator
)
print('Baseline Training Completed!')


In [ ]:
# Step 7: Plot Accuracy & Loss Progression Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history_baseline.history['accuracy'], label='Train Acc', marker='o')
ax1.plot(history_baseline.history['val_accuracy'], label='Val Acc', marker='s')
ax1.set_title('Baseline CNN - Accuracy Progression')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_baseline.history['loss'], label='Train Loss', marker='o', color='red')
ax2.plot(history_baseline.history['val_loss'], label='Val Loss', marker='s', color='orange')
ax2.set_title('Baseline CNN - Loss Progression')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Step 8: Evaluate Baseline Validation Metrics
val_loss, val_acc = baseline_model.evaluate(valid_generator)
print(f'Baseline Validation Accuracy: {val_acc * 100:.2f}%')
print(f'Baseline Validation Loss:     {val_loss:.4f}')


In [ ]:
# Step 9: Save Baseline Model and History
baseline_model.save('baseline_model.h5')
with open('baseline_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history_baseline.history.items()}, f)

print('Saved baseline_model.h5 successfully!')
print('Saved baseline_history.json successfully!')


### Summary of Baseline Model Development
- Custom 3-block CNN with GlobalAveragePooling2D trained for 10 epochs.
- Model weights saved as `baseline_model.h5`.

👉 Now open `model_finetuning.ipynb` to build and fine-tune MobileNetV2!